# RAG Chain

VectorDB, OpenAI API 연동 RAG Chain 구현

> Document (page_content, metadata) → RecursiveCharacterTextSplitter → HuggingFaceEmbeddings 
> 
> → Chroma.from_documents → **<font color=gold>.connect / .excute / as_retriever</font>**

### 환경 구축

In [6]:
# API Key 가져오기 (서비스 개시 전, Github secrets key로 전환)

import os
import pandas as pd
import sqlite3

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_chroma import Chroma
from pathlib import Path
from pydantic import BaseModel, Field



load_dotenv('.env')     # 같은 폴더의 .env
load_dotenv('../.env')  # 상위 폴더의 .env

if not os.getenv('OPENAI_API_KEY') :
    raise RuntimeError(
        'OPENAI_API_KEY를 찾지 못했습니다.\n'
        '다시 확인해 주세요.'
    )
else :
    model = ChatOpenAI(model='gpt-4o-mini', temperature=0, timeout=60)  # 서비스 개시 전, temperature 변경
    print("OpenAI API Key 확인 하고 'MODEL' 생성\n", model.get_graph)

OpenAI API Key 확인 하고 'MODEL' 생성
 <bound method Runnable.get_graph of ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.3'}}, output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002395F09DA60>, async_client=<openai.resources.chat.completions.completions.AsyncCompleti

### ChromaDB CONNECTION

In [3]:
DB_PATH = Path('../chroma_db') / 'chroma.sqlite3'
DB_PATH.parent.mkdir(exist_ok=True)

_conn = sqlite3.connect(DB_PATH, isolation_level=None)

_ro_conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True, isolation_level=None, check_same_thread=False)

print('데이터베이스 연결 완료', DB_PATH)

데이터베이스 연결 완료 ..\chroma_db\chroma.sqlite3


### QUERY 결과 → DataFrame 변환

In [4]:
def run_query(sql) :
    """SELECT 결과를 DataFrame 으로 변환"""
    cur = _conn.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


### QUERY TEST

In [5]:
display(run_query('select * from embeddings'))

,id,segment_id,embedding_id,seq_id,created_at
0,1,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_0,1,2026-08-23 04:41:47
1,2,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_1,2,2026-08-23 04:41:47
2,3,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_2,3,2026-08-23 04:41:47
3,4,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_3,4,2026-08-23 04:41:47
4,5,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_4,5,2026-08-23 04:41:47
...,...,...,...,...,...
3689,3690,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_3689,3690,2026-08-23 04:41:50
3690,3691,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_3690,3691,2026-08-23 04:41:50
3691,3692,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_3691,3692,2026-08-23 04:41:50
3692,3693,206e0893-8dfc-4eab-81df-cf2ce131fe7d,chunk_3692,3693,2026-08-23 04:41:50


### ChromaDB's Schema prompt

In [ ]:
# 모델은 데이터베이스를 볼 수 없습니다 -- 이 글이 모델이 아는 전부입니다.
schema_prompt = """너는 사내 비품 데이터베이스를 조회해 답하는 도우미다.
아래 표만 존재한다. 반드시 run_select 도구로 SQL 을 실행해 확인한 값으로 답한다.

표 구조(sqlite):
  hd_item(item_id text, item_name text, category text, unit_price int, stock int)
    -- 비품 목록. unit_price 는 단가(원), stock 은 현재 재고 수량.
  hd_order(order_id int, item_id text, quantity int, dept text, order_date text)
    -- 부서별 주문 내역. dept 는 주문한 부서 이름.
  hd_order.item_id 는 hd_item.item_id 를 가리킨다.

규칙:
- SELECT 한 문장만 만든다.
- 결과를 사람이 읽을 한국어 문장으로 정리해 답한다."""

print(schema_prompt)

### 구조화된 출력

In [7]:
class SqlForm(BaseModel) :
    """자연어 질문에 대한 결과를 구조화된 형식으로 받는다."""
    sql : str = Field(description = '실행할 SELECT문 표시')
    reason : str = Field(description = 'Query 실행 결과에 대한 이유를 한 문장으로 표시')
    tables : list[str] = Field(description = 'Query가 사용하는 표 이름 목록 표시')

print('스키마 필드 : ', list(SqlForm.model_fields))

스키마 필드 :  ['sql', 'reason', 'tables']


### OpenAI API + BaseModel

In [ ]:
query_result = model.with_structured_output(SqlForm).invoke()